## Question 1 - *What is the distribution of the machines according to their CPU capacity?*

In *machine_events* we can find information about the machines of the system. <br>
Every row of the csv file contains an event recorded in a certain timestamp for a specific machine identified by its unique id. <br>
The information about a machine's capacity can be found in the fifth field, where the values are normalized between 0 and 1 in proportion to the maximum CPU capacity. <br>
For the distribution of the machines according to CPU, we need to take one single event that represents a unique machine. Otherwise, we risk to perturbate the real distribution by the occurrencies of events of some machines that change state frequently. <br>
More explanation can be found in the code comments.



In [1]:
import sys
from pyspark import SparkContext

# Initialising Spark with 1 worker thread
sc = SparkContext("local[1]")

machine_events = sc.textFile("./data/machine_events/part-00000-of-00001.csv.gz")

# Exploring the dataset of machine_events
print("\n" + "="*80)
print(f"Number of machine_events records: {machine_events.count()}")
print(f"First line : ")
print({machine_events.first()})
print(f"Type of the elements : {type(machine_events.first())}")

machine_CPUs_values = (
    machine_events
    .map(lambda line: line.split(",")[4])   # extract CPUs column
    .distinct()
)

print("\n" + "="*80)
print(f"Number of distinct values for CPUs: {machine_CPUs_values.count()}")
print("Values of CPUs:")
print(machine_CPUs_values.take(5))

distribution_machine_CPUs = (
    machine_events
    .map(lambda line: (line.split(",")[1],line.split(",")[4]))  # each element (machine_ID,capacity_CPU)
    .reduceByKey(lambda x1, w2: x1)                             # keeps only the first distinct machine_ID
    .map(lambda x: (x[1],1))                                    # only keep the CPUs for the distribution
    .reduceByKey(lambda a, b: a + b)
)

print("\n" + "="*80)
print("Distribution of the machines according to CPUs:")
print(distribution_machine_CPUs.take(5))



Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/14 22:58:40 WARN Utils: Your hostname, im2ag-mandelbrot, resolves to a loopback address: 127.0.1.1; using 152.77.81.20 instead (on interface ens18)
26/01/14 22:58:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/14 22:58:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Number of machine_events records: 37780
First line : 
{'0,5,0,HofLGzk1Or/8Ildj2+Lqv0UGGvY82NLoni8+J/Yy0RU=,0.5,0.2493'}
Type of the elements : <class 'str'>



Number of distinct values for CPUs: 4
Values of CPUs:
['0.5', '0.25', '1', '']

Distribution of the machines according to CPUs:
[('0.5', 11632), ('0.25', 123), ('1', 796), ('', 32)]
